# Notebook A: long-form Hindi answers, Aya Expanse 8B vs Llama-3.1-8B (paper setup, Obeso et al. 2025)

Both models answer the same Hindi LongFact / LongFact++ prompts from `<you>/longfact-hindi-paper`, using the paper's generation settings:
single turn, temperature 0.1, and up to 2,048 new tokens.
The two models run **at the same time, one per T4**.
Each model gets its own public dataset in the paper's format, `<you>/longfact-hi-<model>`.
The dataset is uploaded every `CHUNK` answers. If Kaggle stops, Run all again and finished chunks are skipped.
Notebook B then labels these answers, and its last cell gives each model's hallucination rate.

**Kaggle:** Accelerator **GPU T4 x2**, **Internet on**, Secrets → **`HF_TOKEN`**: a **write** token of the account in `HF_USER` (the account that receives the datasets).

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U datasets huggingface_hub
import shutil
assert shutil.which("ollama"), "Ollama did not install - read the output above (is Internet on?)"
!nvidia-smi -L

In [ ]:
# ---------------- settings ----------------
HF_USER = "shreshthamodi02"                  # account that holds longfact-hindi-paper and receives the outputs
MODELS = {                                   # Ollama tag -> HF model (8-bit, closest to the paper's runs)
    "llama3.1:8b-instruct-q8_0": "meta-llama/Llama-3.1-8B-Instruct",
    "aya-expanse:8b-q8_0":       "CohereLabs/aya-expanse-8b",
}
SPLITS = ["test"]             # the paper's test prompts; add "train" later for probe training data
N_PER_SOURCE = 150            # per split, from LongFact and from LongFact++ (Hindi test has 989 / 335); None = all
TEMPERATURE = 0.1             # paper, Appendix C
MAX_NEW_TOKENS = 2048         # paper, Appendix C
NUM_CTX = 4096
CHUNK = 20                    # answers per uploaded file (checkpoint)
WORKERS = 4                   # parallel requests per model
SEED = 0

In [ ]:
import json, os, random, re, subprocess, threading, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset
from huggingface_hub import HfApi
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=HF_TOKEN)              # HF_TOKEN must be a WRITE token of HF_USER
USER = HF_USER
me = api.whoami()
assert USER in [me["name"]] + [o["name"] for o in me.get("orgs", [])], (
    f"HF_TOKEN belongs to '{me['name']}', which cannot upload to '{USER}'. Use a write token created while logged in as '{USER}'.")
log = lambda m: print(time.strftime("%H:%M:%S"), m, flush=True)
slug = lambda s: re.sub(r"[^a-zA-Z0-9.]+", "-", s).strip("-").lower()

# ---- Hindi prompts (same rows for both models) ----
prompts = []
for split in SPLITS:
    for short in ["longfact", "longfact-augmented"]:
        rows = list(load_dataset(f"{USER}/longfact-hindi-paper", data_files={split: f"data/{split}-{short}.jsonl"}, split=split))
        for i, r in enumerate(rows):
            r["prompt_id"] = f"{split}-{short}-{i:05d}"
        random.Random(SEED).shuffle(rows)
        rows = rows[:N_PER_SOURCE] if N_PER_SOURCE else rows
        prompts += rows
        log(f"{split}/{short}: {len(rows)} prompts")

# ---- one Ollama server per model, each on its own GPU ----
n_gpu = max(1, len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()))
HOSTS = {}
for i, model in enumerate(MODELS):
    host = f"127.0.0.1:{11434 + i}"
    env = dict(os.environ, OLLAMA_HOST=host, CUDA_VISIBLE_DEVICES=str(i % n_gpu), OLLAMA_NUM_PARALLEL=str(WORKERS))
    subprocess.Popen([shutil.which("ollama"), "serve"], env=env, stdout=open(f"ollama-{i}.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        try: urllib.request.urlopen(f"http://{host}"); break
        except Exception: time.sleep(2)
    subprocess.run([shutil.which("ollama"), "pull", model], env=env, check=True, capture_output=True)
    HOSTS[model] = host
    log(f"{model} on GPU {i % n_gpu} at {host}")

def chat(model, messages, max_tokens=MAX_NEW_TOKENS):
    body = {"model": model, "messages": messages, "stream": False, "keep_alive": "60m",
            "options": {"temperature": TEMPERATURE, "num_predict": max_tokens, "num_ctx": NUM_CTX}}
    req = urllib.request.Request(f"http://{HOSTS[model]}/api/chat", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=1800))["message"]["content"].strip()

for model in MODELS:
    log(f"{model}: {chat(model, [{'role': 'user', 'content': 'नमस्ते! एक छोटे वाक्य में जवाब दें।'}], 40)!r}")
    print(subprocess.run([shutil.which("ollama"), "ps"], env=dict(os.environ, OLLAMA_HOST=HOSTS[model]),
                         capture_output=True, text=True).stdout)   # PROCESSOR should say 100% GPU

In [ ]:
def run(model):
    repo_id = f"{USER}/longfact-hi-{slug(model)}"
    api.create_repo(repo_id, repo_type="dataset", private=False, exist_ok=True)
    done = set(api.list_repo_files(repo_id, repo_type="dataset"))
    log(f"[{model}] -> https://huggingface.co/datasets/{repo_id}")

    def answer(p):
        conv = [{"role": "user", "content": p["prompt_hi"]}]
        conv.append({"role": "assistant", "content": chat(model, conv)})
        return {"subset": p["subset"], "model": MODELS[model], "conversation": conv, "canary": p["canary"],
                "split": p["split"], "lang": "hi", "prompt_id": p["prompt_id"], "ollama_tag": model}

    for c in range(0, len(prompts), CHUNK):
        name = f"data/{c // CHUNK:05d}.jsonl"
        if name in done:
            continue
        t = time.time()
        with ThreadPoolExecutor(WORKERS) as pool:
            rows = list(pool.map(answer, prompts[c:c + CHUNK]))
        local = f"work/{slug(model)}/{name}"
        os.makedirs(os.path.dirname(local), exist_ok=True)
        with open(local, "w", encoding="utf-8") as f:
            f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
        api.upload_file(path_or_fileobj=local, path_in_repo=name, repo_id=repo_id, repo_type="dataset", commit_message=f"add {name}")
        log(f"[{model}] {name}: {len(rows)} answers in {time.time() - t:.0f}s [{min(c + CHUNK, len(prompts))}/{len(prompts)}]")
    log(f"[{model}] done")

threads = [threading.Thread(target=run, args=(m,)) for m in MODELS]   # both models at once
for t in threads: t.start()
for t in threads: t.join()
log("All done.")